# B04 · S5 — El robot, su cinemática y sus problemas

**Objetivo (RA4-a/b):** modelar un brazo como cadena cinemática y entender por qué la **cinemática inversa** es el problema difícil: múltiples soluciones y singularidades.

> Hilo conductor: **la célula LARA**. Práctica guiada de la S5 de los [apuntes](../apuntes.md).

In [ ]:
%pip install roboticstoolbox-python spatialmath-python

## 1. Cinemática directa: brazo plano 3R a mano y con DH

$$x = l_1\cos\theta_1 + l_2\cos(\theta_1{+}\theta_2) + l_3\cos(\theta_1{+}\theta_2{+}\theta_3)$$

Con $l_1=l_2=l_3=1$: ángulos 0 → **(3, 0)**; $\theta_1=90°$ → **(0, 3)**.

In [ ]:
import numpy as np
import roboticstoolbox as rtb
from roboticstoolbox import DHRobot, RevoluteDH

brazo = DHRobot([
    RevoluteDH(a=1.0),
    RevoluteDH(a=1.0),
    RevoluteDH(a=1.0),
], name="Brazo3R")

print("FK [0,0,0]      ->", np.round(brazo.fkine([0, 0, 0]).t, 3).flatten()[:2])
print("FK [90,0,0]     ->", np.round(brazo.fkine([np.pi/2, 0, 0]).t, 3).flatten()[:2])
print("FK [90,90,0]    ->", np.round(brazo.fkine([np.pi/2, np.pi/2, 0]).t, 3).flatten()[:2])

## 2. Puma 560: la tabla DH resuelta

In [ ]:
puma = rtb.models.DH.Puma560()
pose = puma.fkine([0, 0.2, 0.3, 0.4, 0.5, 0.6])
print("Pose del efector (t):\n", np.round(pose.t, 3))

## 3. Cinemática inversa: múltiples soluciones y convergencia

In [ ]:
robot = rtb.models.Panda()
pose_objetivo = robot.fkine([0, -0.8, 0.8, 0, 0.8, 0, 0])

sol = robot.ikine_LM(pose_objetivo)
print("¿ha convergido?:", sol.success)
print("q encontrado:  ", np.round(sol.q, 3))
print("q original:    ", [0, -0.8, 0.8, 0, 0.8, 0, 0])

# error de posicion entre la pose pedida y la reconstruida
error = np.linalg.norm(robot.fkine(sol.q).t - pose_objetivo.t)
print("Error de posicion:", round(float(error), 5), "m")

## 4. Singularidades: cuando el jacobiano pierde rango

En una singularidad, las velocidades articulares necesarias tienden a infinito. Con el Panda (o el Puma 560), prueba configuraciones con el brazo **estirado** y observa qué devuelve `ikine_LM` y el **índice de manipulabilidad** (Yoshikawa).

In [ ]:
# Manipulabilidad: cuanto mas cerca de 0, peor (mas cerca de una singularidad)
import roboticstoolbox.tools as tools

q_extendido = np.zeros(7)          # Panda estirado
print("manipulabilidad (estirado):", round(float(robot.manipulability(q_extendido)), 5))
q_flexionado = np.array([0.5, -0.6, 0.3, -0.7, 0.2, 0.9, 0.3])
print("manipulabilidad (flexionado):", round(float(robot.manipulability(q_flexionado)), 5))

## Actividad A1 (autónoma, entregable)

1. Brazo **RR** planar ($l_1=1, l_2=1$): FK para 4 configuraciones y comparación con la fórmula a mano.
2. Para la pose **(1, 1)**, encuentra **dos soluciones** de IK (codo arriba / codo abajo) y verifícalas con `fkine`.
3. Panda: FK + IK para una pose arbitraria; reporta convergencia y error.

Pista: en el brazo RR, el codo arriba y el codo abajo son simétricos respecto a la línea base-efector:

$$x = l_1\cos\theta_1 + l_2\cos(\theta_1{+}\theta_2), \qquad y = l_1\sin\theta_1 + l_2\sin(\theta_1{+}\theta_2)$$

In [ ]:
# TODO A1: brazo RR, FK a mano vs roboticstoolbox, dos soluciones de IK
brazo_rr = DHRobot([RevoluteDH(a=1.0), RevoluteDH(a=1.0)], name="RR")
...